# FootballAnalytics: Ballkandidaten aus dem 1080p-Video

Das Notebook erkennt den Ball mit niedriger Schwelle auf der T4. Falsche Kandidaten werden später lokal über Platzgeometrie und zeitliche Konsistenz entfernt.

Vorher in Google Drive ablegen: `Meine Ablage/FootballAnalytics/input/1080hp_day1.mp4`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Keine GPU aktiv: Laufzeit > Laufzeittyp ändern > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/FootballAnalytics')
VIDEO_NAME = '1080hp_day1.mp4'
REPO_URL = 'https://github.com/timg4/FootballAnalytics.git'
BRANCH = 'main'
INPUT_VIDEO = DRIVE_ROOT / 'input' / VIDEO_NAME
RESULTS_DIR = DRIVE_ROOT / 'results' / '1080hp_day1'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_VIDEO.exists(), f'Video fehlt: {INPUT_VIDEO}'
print('Video:', INPUT_VIDEO, f'{INPUT_VIDEO.stat().st_size / 1e9:.2f} GB')

In [ ]:
import os, shutil, subprocess
WORK = Path('/content/FootballAnalytics')
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements.txt')], check=True)
os.chdir(WORK)
print('Code-Version:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
LOCAL_VIDEO = Path('/content') / VIDEO_NAME
if not LOCAL_VIDEO.exists() or LOCAL_VIDEO.stat().st_size != INPUT_VIDEO.stat().st_size:
    shutil.copy2(INPUT_VIDEO, LOCAL_VIDEO)
print('Lokale Kopie:', LOCAL_VIDEO, f'{LOCAL_VIDEO.stat().st_size / 1e9:.2f} GB')

In [ ]:
BALL_CSV = WORK / 'data/output/video_project_1080_ball_candidates.csv'
cmd = [
    'python', 'src/detect_ball.py', str(LOCAL_VIDEO),
    '--model', 'yolo11s.pt', '--imgsz', '1920', '--conf', '0.03',
    '--device', '0', '--batch-size', '4',
    '--start', '10', '--end', '25160', '--frame-offset', '-10',
    '--output', str(BALL_CSV),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
target = RESULTS_DIR / BALL_CSV.name
shutil.copy2(BALL_CSV, target)
print('Gesichert:', target, f'{target.stat().st_size / 1e6:.1f} MB')
print('Zeilen inklusive Kopf:', sum(1 for _ in open(BALL_CSV, encoding='utf-8')))
print('Fertig. CSV aus Drive nach data/output herunterladen.')